# COMMENT WHY SUSPENDED

It seems this dataset is a very weird case of task and how to setup, we need to make some big judgment calls. This seems like a big task with a lot of discussion and maybe it takes too much time for now.

THe text below contains a lot of information but it is not fully re-written after getting all the insights.

### Dataset and Task Metadata

In [1]:
from data_foundry.schema import DatasetMetadata, PredictiveMLTaskMetadata

dataset_mold = DatasetMetadata(
    unique_name="ashrae_energy_predictor",
    dataset_year="2019",
    domain_str="environmental science & climate",
    # Data Source
    dataset_source="Kaggle",
    original_dataset_source_download_link="https://www.kaggle.com/competitions/ashrae-energy-prediction",
    download_description="""
We download the data from Kaggle.

kaggle competitions download -c ashrae-energy-prediction && unzip ashrae-energy-prediction.zip -d data_files && rm ashrae-energy-prediction.zip
mkdir -p local-data-warehouse/ashrae_energy_predictor && mv data_files local-data-warehouse/ashrae_energy_predictor/
""",
    # References
    academic_reference_bibtex=r"""@misc{Howard2019ASHRAEGreatEnergyPredictorIII,
  author = {Addison Howard and Chris Balbach and Clayton Miller and Jeff Haberl and Krishnan Gowri and Sohier Dane},
  title  = {ASHRAE - Great Energy Predictor III},
  year   = {2019},
  howpublished = {\url{https://kaggle.com/competitions/ashrae-energy-prediction}},
  note   = {Kaggle competition}
}
""",
    academic_reference_bibtex_key="Howard2019ASHRAEGreatEnergyPredictorIII",
    license="Kaggle Competition Rules",
    data_tags=["Non-IID", "Grouped", "Temporal"],
    curation_comments="""
We start with the data from Kaggle and try to reproduce the data from the winning solutions. Their code is not public, as the competition organizer did not want to share it.
So instead, we build on top of other notebooks and kernels, that we can find online. But it seems, the competition had a lot of diverging solutions, mostly as the competition had a data leak and the data had many outlier and noisy sensor readings. We do our best here, to create a version of the dataset that has only the most basic preprocessing needed for training a tabular model on the data.
Moreover, the task was solvable by training one model per subset of the data. Most winning solutions used several models trained on all data and subsets that were ensembled. Here, we create the task to train one model across all data.
Lastly, we are fully aware that this is a in nature a forecasting task. One predicts the energy meter reading of other buildings in the future. We still treat this competition as a tabular ML task, because all top solutions solved it via tabular ML and not forecasting models.

NVM, found code of top solutions. they really fit one model per type. This gets super complex, unsure how to really decide on this dataset. Suspend for now....



- We log1p transform the target
""",
)
task_mold = PredictiveMLTaskMetadata(
    target_column_name="x",
    problem_type="regression",
    objective_metric_name="rmse", # rmsle in the competition, but we use rmse as we already log-transform the target
    stratify_on="",
    group_on="",
    time_on=""
)

## Preprocessing

In [ ]:
import pandas as pd
import numpy as np

df = pd.read_csv(dataset_mold.path / "xxx.csv")
print("Loaded data shape:", df.shape)

In [ ]:
# Use if needed to get see all cols of pandas dataframes
# pd.set_option("display.max_rows", None)
# pd.set_option("display.max_columns", None)
# pd.set_option("display.width", None)
# pd.set_option("display.max_colwidth", None)

In [ ]:
# https://github.com/buds-lab/ashrae-great-energy-predictor-3-solution-analysis/tree/master/solutions/rank-1

# Maybe just follow this: https://www.kaggle.com/code/purist1024/ashrae-simple-data-cleanup-lb-1-08-no-leaks / https://www.kaggle.com/code/isaienkov/lightgbm-fe-1-19?scriptVersionId=22543478 https://www.kaggle.com/code/rohanrao/ashrae-half-and-half https://www.kaggle.com/code/aitude/ashrae-kfold-lightgbm-without-leak-1-08
# Maybe add some outlier removal and detection from here: \
#   - https://www.kaggle.com/code/corochann/ashrae-training-lgbm-by-meter-type

# https://arxiv.org/pdf/2106.13475
# https://arxiv.org/abs/2007.06933
# https://github.com/buds-lab/ashrae-great-energy-predictor-3-solution-analysis
# https://github.com/buds-lab/ashrae-great-energy-predictor-3-overview-analysis

# The last thing has code, great! But note, the weather data is from the future... so it is not forecasting but non-iid tabular data??

## Data Checks

In [ ]:
from data_foundry import dataset_checks
df_head, summary, numeric_stats, cat_stats, target_df = dataset_checks.run_all_checks(
    data=df,
    classification=task_mold.is_classification,
    target_feature=task_mold.target_column_name,
    print_report=False, # In notebook...
)

In [ ]:
# Sample Rows
df_head

In [ ]:
# Feature Summary
summary

In [ ]:
# Numeric Feature Statistics
numeric_stats

In [ ]:
# Categorical Feature Statistics
cat_stats

In [ ]:
# Target Distribution
target_df

## Task Curation

In [ ]:
from data_foundry.curation_recommendations import get_recommended_splits_dimensions

n_repeats, n_splits, none_or_test_size = get_recommended_splits_dimensions(dataset=df)
print(f"Recommended IID splits: n_repeats={n_repeats}, n_splits={n_splits}, test_size={none_or_test_size}")

In [ ]:
from data_foundry.schema import PredictiveMLSplitsMetadata
from data_foundry.curation_recommendations import get_recommended_iid_splits

# The structure of splits is:
# splits = {
#     repeat_i: {
#         fold_i: (train_idx, test_idx),
#     }
# }

splits_mold = PredictiveMLSplitsMetadata(
    splits_comment="Default splits for IID data.",
    splits=get_recommended_iid_splits(
        dataset=df,
        n_repeats=n_repeats,
        n_splits=n_splits,
        test_size=none_or_test_size,
        stratify_on=task_mold.stratify_on,
    ),
)

## Export

In [ ]:
from data_foundry.curation_container import CuratedContainer
curated_data = CuratedContainer(
    dataset=df,
    dataset_metadata=dataset_mold,
    task_metadata=task_mold,
    experiment_metadata=splits_mold,
 )
curated_data.save()
print(curated_data.uuid)
print(curated_data.checksum)